In [1]:
import pandas as pd
import numpy as np
import json

## Define the metadata

In [2]:
organism = "homo_sapiens"
cell_source = "adipose"
cell_state = None

reference = "GRCh38"
paper_doi = "https://www.nature.com/articles/s41590-021-00922-4"
table_link = "https://static-content.springer.com/esm/art%3A10.1038%2Fs41590-021-00922-4/MediaObjects/41590_2021_922_MOESM2_ESM.xlsx"

# don't include in header
table_name = "41590_2021_922_MOESM2_ESM.xlsx"
table_name = "paper_degs.xlsx" # local name # UPDATE THIS TO BE SPECIFIC FOR HILDRETH

## Read in file

In [3]:
excel = pd.read_excel(table_name, sheet_name= "Cluster_Markers_CD45PosNeg", skiprows = 2)

df = excel

In [4]:
df.head()

,p_val,avg_logFC,pct.1,pct.2,p_val_adj,cluster,gene
0,0.000000e+00,0.911908,0.269,0.014,0.000000e+00,CD8 gdT,LINC02446
1,0.000000e+00,0.905986,0.464,0.043,0.000000e+00,CD8 gdT,LEF1
2,0.000000e+00,0.748453,0.301,0.016,0.000000e+00,CD8 gdT,FXYD2
3,0.000000e+00,0.727598,0.262,0.006,0.000000e+00,CD8 gdT,ZNF683
4,4.540614e-241,0.962575,0.329,0.031,1.242811e-236,CD8 gdT,KLRC2


## Perform the filtering

In [5]:
col_pval = "p_val_adj"
col_lfc = "avg_logFC"
col_gene = "gene"
col_ct = "cluster" # says cluster in table, but before this had "celltype"
col_rank = "pct.1" # set to None if pct. 1DNE

In [6]:
"""
min_mean = 100
max_pval = 1e-10
min_lfc = 2.2
max_gene_shares = 2
max_per_celltype = 20
"""

min_mean = 0
max_pval = 1
min_lfc = 0
max_gene_shares = 100
max_per_celltype = 100

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)]

if col_rank:
    m = m.sort_values(col_rank, ascending = True)

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [7]:
markers

,p_val,avg_logFC,pct.1,pct.2,p_val_adj,cluster,gene
22942,8.054062e-149,0.797017,0.652,0.414,2.204477e-144,Endothelial,SYNE2
22512,0.000000e+00,2.109642,0.653,0.136,0.000000e+00,Endothelial,ID1
23410,1.222424e-17,0.268743,0.658,0.654,3.345897e-13,Endothelial,ZFAND5
22560,0.000000e+00,1.460513,0.660,0.146,0.000000e+00,Endothelial,CRIP2
22519,0.000000e+00,1.993474,0.661,0.021,0.000000e+00,Endothelial,FLT1
...,...,...,...,...,...,...,...
19373,7.675226e-208,0.174211,1.000,0.997,2.100786e-203,APC,TMSB4X
2374,1.665137e-41,0.388889,1.000,0.995,4.557646e-37,ILCs,RPL32
112,1.600780e-47,0.305025,1.000,0.988,4.381494e-43,CD8 gdT,RPS25
2383,2.731580e-38,0.427961,1.000,0.995,7.476609e-34,ILCs,RPL28


In [8]:
markers.groupby(col_ct)[col_rank].mean().sort_values()

cluster
Endothelial        0.79105
Smooth muscle      0.88315
mNK                0.89646
PVM                0.92945
Preadipocyte       0.96079
Cytotoxic CD8 T    0.96487
NK-like            0.96607
Naive CD8 T        0.97378
Myeloid-like       0.97581
APC                0.97784
Treg               0.97967
B cell             0.98657
Naive CD4 T        0.98970
ncMo               0.99136
CD8 gdT            0.99302
cDC1               0.99439
cDC2B              0.99475
MAIT               0.99507
ILCs               0.99545
Name: pct.1, dtype: float64

In [9]:
markers[col_ct].value_counts()

cluster
Endothelial        100
Myeloid-like       100
cDC2B              100
MAIT               100
ncMo               100
ILCs               100
CD8 gdT            100
Naive CD4 T        100
B cell             100
Treg               100
Smooth muscle      100
APC                100
Preadipocyte       100
Naive CD8 T        100
NK-like            100
Cytotoxic CD8 T    100
PVM                100
mNK                100
cDC1               100
Name: count, dtype: int64

## Find Ensembl ID


In [10]:
import sys
import time

from urllib.parse import urlparse, urlencode
from urllib.request import urlopen, Request
from urllib.error import HTTPError

In [11]:
class EnsemblRestClient(object):
    def __init__(self, server='http://rest.ensembl.org', reqs_per_sec=5):
        self.server = server
        self.reqs_per_sec = reqs_per_sec
        self.req_count = 0
        self.last_req = 0

    def perform_rest_action(self, endpoint, hdrs=None, params=None):
        if hdrs is None:
            hdrs = {}

        if 'Content-Type' not in hdrs:
            hdrs['Content-Type'] = 'application/json'

        if params:
            endpoint += '?' + urlencode(params)

        data = None

        # check if we need to rate limit ourselves
        if self.req_count >= self.reqs_per_sec:
            delta = time.time() - self.last_req
            if delta < 1:
                time.sleep(1 - delta)
            self.last_req = time.time()
            self.req_count = 0
        
        try:
            request = Request(self.server + endpoint, headers=hdrs)
            response = urlopen(request)
            content = response.read()
            if content:
                data = json.loads(content)
            self.req_count += 1

        except HTTPError as e:
            # check if we are being rate limited by the server
            if e.code == 429:
                if 'Retry-After' in e.headers:
                    retry = e.headers['Retry-After']
                    time.sleep(float(retry))
                    self.perform_rest_action(endpoint, hdrs, params)
            else:
                sys.stderr.write('Request failed for {0}: Status code: {1.code} Reason: {1.reason}\n'.format(endpoint, e))
           
        return data

    def get_id(self, species, symbol):
        genes = self.perform_rest_action(
            endpoint='/xrefs/symbol/{0}/{1}'.format(species, symbol), 
            params={'object_type': 'gene'}
        )
        if genes:
            stable_id = genes[0]['id']
            return stable_id
        else:
            return "gene not found"

In [12]:
def run(symbol):
    client = EnsemblRestClient()
    gene_id = client.get_id('human', symbol)
    return gene_id

## Convert to evidence json


In [13]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "gene"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["gene_id"] = df["gene"]
df["gene_id"] = df["gene_id"].apply(run)
save = df.to_dict(orient = "records")

InvalidURL: URL can't contain control characters. '/xrefs/symbol/human/2020-09-07 00:00:00?object_type=gene' (found at least ' ')

In [14]:
save

[{'cell_type_label': 'B cell',
  'gene': 'IGHG3',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': 'ENSG00000211897'},
 {'cell_type_label': 'B cell',
  'gene': 'IGHG1',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': 'ENSG00000211896'},
 {'cell_type_label': 'B cell',
  'gene': 'IGLC3',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': 'ENSG00000211679'},
 {'cell_type_label': 'B cell',
  'gene': 'IGHA1',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': 'ENSG00000211895'},
 {'cell_type_label': 'B cell',
  'gene': 'IGLC2',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': 'ENSG00000211677'},
 {'cell_type_label': 'B cell',
  'gene': 'JCHAIN',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None,
  'gene_id': 'ENSG00000132465'},
 {'cell_type_label': 

## Save evidence

In [15]:
with open("evidence_test.json", "w") as f:
    json.dump(save, f, indent = 4) 